# Stage 6b — **TEST ONLY**: run 2 — `llada-moe-lora-run2-NYC-hyperbolic-ckpt` (checkpoint `best/`)

Evaluation-only version of the run-2 training notebook. It rebuilds the *identical* data pipeline
(same `/kaggle/input` paths, same 70/10/20 per-user chronological re-split, same causal-profile
prompt, same frozen `W_POI` injection), **pulls the finetuned checkpoint from Hugging Face**
(`yosrr12/llada-moe-lora-run2-NYC-hyperbolic-ckpt`, subfolder `best/`), and runs the restricted-logit ranking eval →
**Acc@1 / Acc@5 / Acc@10 / MRR** on the test (and val) split. No training happens here.

**Before running:**
1. Attach the **same Kaggle datasets** as the training notebook (the stage-0 CSVs
   `train/val/test_NYC.csv`, `poi_metadata_NYC.csv`, `vocab.pkl`, and the embeddings `.npy`).
2. Add your **`HF_TOKEN`** via Add-ons → Secrets (the checkpoint repo is private).
3. GPU on (T4 is enough — eval is one forward pass per example, no gradients).

⚠ `EMB_CONDITION` / `EMB_FILE` in the config cell **must match what this checkpoint was trained
with** — the tied scoring path ranks against the frozen `W_POI` built from them; a mismatch
silently wrecks the tied term and the metrics.


## 0 · Environment  
*§0a and the pip pin are copied unchanged from the training notebook — same kernel-restart rules apply.*

In [ ]:
# ── 0a · Authenticate + pre-download the weights BEFORE the transformers pin ──
# ⚠ RUN THIS IN A FRESH KERNEL (Session → Restart), BEFORE the pip cell.
#   HF_HUB_DISABLE_XET is read by huggingface_hub AT IMPORT TIME, so it must be set before the
#   first `import huggingface_hub` in the process — hence the restart + this-cell-first rule.
#
# WHY THIS CELL EXISTS — the run-2 "model loading takes forever" bug. §6's load cell is
# byte-identical to run 1, so the cost is in the DOWNLOAD, not the loading code. Three things
# were wrong, all fixed here:
#
#   (1) NO HF TOKEN → anonymous, shared-IP rate limits on Kaggle.  [FIXED: Kaggle Secrets]
#   (2) hf_xet STALLING → the download starts at ~40 MB/s and then wedges (observed: stuck at
#       ~200 MB of 14.7 GB, while disk was 87% free — so neither disk nor throttling). Xet
#       reassembles chunks client-side and Kaggle only gives ~4 vCPUs. We disable it and use
#       plain authenticated HTTP, which is boring and reliable.  [FIXED: DISABLE_XET]
#   (3) THE PIN DOWNGRADES huggingface_hub — the image ships 1.11.0, but transformers==4.46.3
#       requires <1.0, so the pip cell knocks it across the 1.0 boundary. Fetching the snapshot
#       HERE, before the pin, means §6 loads from local disk and never touches the network.
import os, sys

DISABLE_XET = True          # flip to False to A/B the Xet transport
MAX_WORKERS = 4             # Kaggle has ~4 vCPUs; 8 thrashes

if DISABLE_XET:
    os.environ["HF_HUB_DISABLE_XET"] = "1"
if "huggingface_hub" in sys.modules:
    print("⚠ huggingface_hub is ALREADY imported — HF_HUB_DISABLE_XET is read at import time and\n"
          "  will NOT take effect. Session → Restart, then run this cell FIRST.\n")

import time, importlib, importlib.metadata as md, huggingface_hub
from huggingface_hub import snapshot_download

MODEL_NAME    = "inclusionAI/LLaDA-MoE-7B-A1B-Instruct"   # keep in sync with the config cell (§1)
MODEL_NAME_0A = MODEL_NAME                                # §1 asserts against this

# ── (1) token: Kaggle Secrets -> env, before any hub call ───────────────────
if not os.environ.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print(f"⚠ could not load HF_TOKEN from Kaggle Secrets ({type(e).__name__}: {e})")

if os.environ.get("HF_TOKEN"):
    from huggingface_hub import whoami
    try:
        print(f"HF auth                   : OK, logged in as {whoami()['name']}")
    except Exception as e:
        print(f"⚠ HF_TOKEN is set but rejected by the Hub: {e}")
else:
    print("HF auth                   : ANONYMOUS → shared-IP rate limits on Kaggle. "
          "Add HF_TOKEN via Add-ons → Secrets.")

# ── (2) transport ───────────────────────────────────────────────────────────
HUB_BEFORE_PIN = huggingface_hub.__version__
try:
    from huggingface_hub.constants import HF_HUB_DISABLE_XET
    _xet_off = bool(HF_HUB_DISABLE_XET)
except Exception:
    _xet_off = False
print(f"huggingface_hub (pre-pin) : {HUB_BEFORE_PIN}")
print(f"transport                 : {'plain HTTP (Xet disabled)' if _xet_off else 'Xet'}  "
      f"max_workers={MAX_WORKERS}")
if DISABLE_XET and not _xet_off:
    print("  ⚠ Xet is still ACTIVE despite DISABLE_XET — the hub was imported too early. Restart.")

# snapshot_download RESUMES: files already complete in the cache are skipped, so re-running
# after a stall costs nothing. Progress bars give live MB/s — a stall is now visible.
_t0 = time.time()
MODEL_PATH = snapshot_download(MODEL_NAME, max_workers=MAX_WORKERS)
_dt = time.time() - _t0

print(f"\nsnapshot ready in {_dt/60:.1f} min ({_dt:.0f}s)")
print(f"MODEL_PATH = {MODEL_PATH}")
!du -shL {MODEL_PATH}

# ── ALSO pre-fetch the finetuned checkpoint (adapter + head) — pre-pin, token already set ──
# trainer_state.pt (~1.3 GB of optimizer state) is deliberately EXCLUDED — eval never needs it.
CKPT_REPO_0A      = "yosrr12/llada-moe-lora-run2-NYC-hyperbolic-ckpt"
CKPT_SUBFOLDER_0A = "best"            # this repo has `best/` (lowest val loss) and `latest/` — default is `best`

_t0 = time.time()
CKPT_ROOT = snapshot_download(
    CKPT_REPO_0A,
    allow_patterns=[f"{CKPT_SUBFOLDER_0A}/adapter_config.json",
                    f"{CKPT_SUBFOLDER_0A}/adapter_model.safetensors",
                    f"{CKPT_SUBFOLDER_0A}/poi_head.pt"],
    max_workers=MAX_WORKERS,
)
CKPT_DIR = os.path.join(CKPT_ROOT, CKPT_SUBFOLDER_0A)
for _f in ("adapter_config.json", "adapter_model.safetensors", "poi_head.pt"):
    assert os.path.isfile(os.path.join(CKPT_DIR, _f)), f"checkpoint file missing: {_f}"
print(f"checkpoint ready in {time.time()-_t0:.0f}s")
print(f"CKPT_DIR = {CKPT_DIR}")
!du -shL {CKPT_DIR}


In [ ]:
# ── Environment: hard-clean install of the pinned stack ────────────────────
# ⚠ §0a (pre-download) MUST have run before this cell — see the assert at the bottom.
# Kaggle's base image now ships a NEWER transformers. Plain `pip install transformers==4.46.3`
# downgrades OVER it and leaves orphan files, producing a MIXED package on disk:
#     ImportError: cannot import name 'GGUF_TENSOR_MAPPING' from 'transformers.integrations'
# (modeling_gguf_pytorch_utils.py is 4.46.3's, integrations/__init__.py is the newer one.)
# pip uninstall does NOT reliably remove those orphans, so we DELETE the package dirs outright.
import glob, shutil, os, sys

!pip uninstall -y -q transformers tokenizers 2>/dev/null

for pat in ("transformers", "transformers-*", "tokenizers", "tokenizers-*"):
    for p in glob.glob(f"/usr/local/lib/python3.12/dist-packages/{pat}"):
        shutil.rmtree(p, ignore_errors=True) if os.path.isdir(p) else os.remove(p)
print("leftover transformers files:", glob.glob("/usr/local/lib/python3.12/dist-packages/transformers*") or "NONE (clean)")

!pip install -q --no-cache-dir "transformers==4.46.3" "bitsandbytes>=0.46.1" "peft==0.13.2" accelerate

# Verify BEFORE any other cell imports transformers. This is the exact import chain that failed.
import transformers, tokenizers
from transformers.modeling_utils import PreTrainedModel          # <- the line that was crashing
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
print(f"OK  transformers={transformers.__version__}  tokenizers={tokenizers.__version__}")
assert transformers.__version__ == "4.46.3", "pin did not take — restart the session and re-run this cell"

# ── DIAGNOSTIC: did the transformers pin drag huggingface_hub down with it? ──
# transformers 4.46.3 requires huggingface-hub<1.0. If the image shipped hub 1.x, pip has just
# DOWNGRADED it — which loses the Xet transport and makes hub downloads crawl. That is precisely
# why §0a pre-fetches the weights: by the time we get here they are already on local disk, so the
# downgrade is HARMLESS. The print below is evidence for the diagnosis, not an error.
import huggingface_hub
_before = globals().get("HUB_BEFORE_PIN", "unknown (§0a not run in this kernel)")
_after  = huggingface_hub.__version__
print(f"huggingface_hub: {_before} (pre-pin) -> {_after} (post-pin)")
if _before != _after:
    print("  ^ CONFIRMED: the pin downgraded the hub — this is what made run 2's download slow.")
    print("    Harmless now: §0a already cached the weights and §6 loads from MODEL_PATH.")

assert "MODEL_PATH" in globals(), (
    "Run §0a (pre-download) FIRST — it must precede this pin, or from_pretrained will download "
    "the 14 GB of shards over the slow downgraded hub."
)

# If this cell RAISES, the kernel now holds broken half-imported modules in sys.modules:
#   Run -> Restart session, then run §0a and this cell FIRST in the fresh kernel.

In [ ]:
# ── GPU sanity check — run this FIRST after any restart, before loading anything ──
# If "Memory-Usage" here is already high (e.g. >1-2 GiB) on what you believe is a
# fresh kernel, the GPU is NOT actually clear. Two likely causes on Kaggle:
#   1. The kernel wasn't really restarted (Session menu -> Restart Session, not just
#      stopping a cell — stopping does not reliably free CUDA memory).
#   2. A SEPARATE Kaggle session is holding the GPU concurrently — most commonly a
#      background "Save Version" / "Save & Run All" commit run. Check
#      kaggle.com -> your notebook -> the Sessions/Versions panel (or account ->
#      Notebooks) for any OTHER active session on this GPU and stop it, then restart
#      this session and re-run.
!nvidia-smi

## 1 · Config

In [ ]:
# ── Config (TEST ONLY) ──────────────────────────────────────────────────────
# Everything here MUST mirror the training notebook's config for the numbers to be meaningful.
import os, glob

DATASET       = "NYC"
DATA_DIR      = "/kaggle/input"          # searched recursively for the files below (same as training)
OUT_DIR       = "/kaggle/working"
MODEL_NAME    = "inclusionAI/LLaDA-MoE-7B-A1B-Instruct"   # 7B total / ~1B ACTIVE
assert MODEL_NAME == globals().get("MODEL_NAME_0A", MODEL_NAME), \
    "MODEL_NAME here disagrees with §0a — re-run §0a after changing it"

# ── checkpoint under test (downloaded in §0a) ───────────────────────────────
CKPT_REPO      = "yosrr12/llada-moe-lora-run2-NYC-hyperbolic-ckpt"
CKPT_SUBFOLDER = "best"                 # this repo has `best/` (lowest val loss) and `latest/` — default is `best`
assert CKPT_REPO == globals().get("CKPT_REPO_0A", CKPT_REPO) and \
       CKPT_SUBFOLDER == globals().get("CKPT_SUBFOLDER_0A", CKPT_SUBFOLDER), \
    "checkpoint repo/subfolder here disagrees with §0a — re-run §0a after changing it"
assert "CKPT_DIR" in globals(), "Run §0a first — it downloads the checkpoint"

# ⚠ MUST MATCH TRAINING for this checkpoint. The tied path ('both'/'tied') scores against the
#   frozen W_POI built from EMB_FILE + EMB_CONDITION + SEED — a mismatch here silently corrupts
#   the tied logits. SCORING_MODE itself is NOT set here: it is read from the checkpoint's
#   poi_head.pt in §9.
EMB_CONDITION = "hyperbolic"             # "hyperbolic" | "euclidean" | "random"
EMB_FILE      = "/kaggle/input/datasets/yosrkharrat/kushflq/poi_hyperbolic_embs.npy"
CURVATURE_C   = 1.0                      # RotH curvatures were initialised at c=1
PROJ_HIDDEN   = None                     # tied-MLP hidden width; None -> H // 2 (as in training)

# ── split — identical to training (run-2 re-split) ─────────────────────────
RESPLIT       = True
TRAIN_FRAC    = 0.70
VAL_FRAC      = 0.10                     # test gets the remaining 0.20

# ── prompt — identical to training ──────────────────────────────────────────
USE_PROFILE   = True
PROFILE_TOP_K = 5
PROFILE_CATS  = 3
PROFILE_HOURS = 3

# LLaDA-MoE mask id (official generate() default mask_id=156895). NOT 126336 (that's 8B-dense).
MASK_TOKEN_ID = 156895

HIST_LEN      = 15
MAX_LEN       = 1024
BATCH_SIZE    = 8                        # eval has no gradients — 16 usually also fits on a T4
SEED          = 42

EVAL_VAL      = True                     # also score the val split (test always runs)

def find(fname):
    """Locate a file under DATA_DIR, tolerating Kaggle's ' (n)' re-upload suffixes."""
    if os.path.isabs(fname) and os.path.exists(fname):
        return fname
    hits = glob.glob(os.path.join(DATA_DIR, "**", fname), recursive=True)
    if not hits:
        stem, ext = os.path.splitext(os.path.basename(fname))
        hits = sorted(glob.glob(os.path.join(DATA_DIR, "**", f"{stem}*{ext}"), recursive=True))
    assert hits, f"File not found under {DATA_DIR}: {fname}"
    print(f"  {fname} -> {hits[0]}")
    return hits[0]


## 2 · Load data & artifacts — *identical paths to training*

In [ ]:
# ── Load data & artifacts ──────────────────────────────────────────────────
import numpy as np, pandas as pd, pickle, torch, random

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

train_df = pd.read_csv(find(f"train_{DATASET}.csv"))
val_df   = pd.read_csv(find(f"val_{DATASET}.csv"))
test_df  = pd.read_csv(find(f"test_{DATASET}.csv"))
meta_df  = pd.read_csv(find(f"poi_metadata_{DATASET}.csv"))

with open(find("vocab.pkl"), "rb") as f:
    vocab = pickle.load(f)

poi_embs = np.load(find(EMB_FILE))
N_POI = len(meta_df)
assert poi_embs.shape[0] == N_POI, f"emb rows {poi_embs.shape[0]} != n_poi {N_POI}"
print(f"POIs: {N_POI} | emb dim: {poi_embs.shape[1]} | condition: {EMB_CONDITION}")

# hour-of-day + weekday for the prompt's temporal context
for df in (train_df, val_df, test_df):
    if "utc_time" in df.columns:
        ts = pd.to_datetime(df["utc_time"], errors="coerce", utc=True)
        df["hour"] = ts.dt.hour.fillna(12).astype(int)
        df["dow"]  = ts.dt.day_name().fillna("Monday")
    else:
        df["hour"] = 12; df["dow"] = "Monday"

poi_cat = meta_df.set_index("poi_idx")["category"].fillna("Venue").to_dict()

## 3 · Re-split 70/10/20 + build (profile, history, target) examples — *identical to training*

Same per-user chronological re-split and the same **strictly causal** profile construction as the
training notebook (running `Counter` over `seq[:i]` only). The re-split is fully deterministic
(stable mergesort, no RNG), so the **test targets produced here are exactly the training run's
test targets**. Only `val` and `test` examples are materialised — no training examples are needed.


In [ ]:
# ── Re-split 70/10/20 + build (profile, history, target) examples ──────────
# VERBATIM from the training notebook (only train_ex construction is skipped — not needed here).
from collections import Counter

for _df, _name in ((train_df, "train"), (val_df, "val"), (test_df, "test")):
    _df["split"] = _name                      # stage-0 labels (kept only for the audit print below)
full_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
_orig_split = full_df["split"].copy()

def resplit_per_user(df, train_frac=TRAIN_FRAC, val_frac=VAL_FRAC):
    """Overwrite df['split'] with a per-user chronological 70/10/20 assignment."""
    df = df.sort_values(["user_id", "utc_time"] if "utc_time" in df.columns else ["user_id"],
                        kind="mergesort").reset_index(drop=True)
    splits = np.empty(len(df), dtype=object)
    for _, idx in df.groupby("user_id", sort=False).indices.items():
        n = len(idx)
        # ceil the train boundary so a 2-check-in user still contributes a train example
        n_tr  = max(1, int(np.ceil(n * train_frac)))
        n_val = int(np.ceil(n * (train_frac + val_frac)))
        n_val = min(max(n_val, n_tr), n)                       # monotone, clipped
        splits[idx[:n_tr]]        = "train"
        splits[idx[n_tr:n_val]]   = "val"
        splits[idx[n_val:]]       = "test"
    df["split"] = splits
    return df

if RESPLIT:
    full_df = resplit_per_user(full_df)
    vc = full_df["split"].value_counts(normalize=True)
    print(f"[re-split] per-user chronological -> "
          f"train={vc.get('train',0):.3f}  val={vc.get('val',0):.3f}  test={vc.get('test',0):.3f} "
          f"(target {TRAIN_FRAC:.2f}/{VAL_FRAC:.2f}/{1-TRAIN_FRAC-VAL_FRAC:.2f})")
    print(f"[re-split] check-ins that changed split vs stage-0: "
          f"{(full_df['split'].values != _orig_split.values).mean():.1%}")

def _profile_from_prefix(poi_counter, cat_counter, hour_counter, n_seen):
    """Snapshot of everything the model is allowed to know about the user BEFORE the target."""
    return dict(
        n_seen   = n_seen,
        top_pois = poi_counter.most_common(PROFILE_TOP_K),     # [(poi_idx, count), ...]
        top_cats = [c for c, _ in cat_counter.most_common(PROFILE_CATS)],
        top_hrs  = [h for h, _ in hour_counter.most_common(PROFILE_HOURS)],
    )

def build_split_examples(full_df, target_split, hist_len=HIST_LEN):
    ex = []
    for uid, g in full_df.groupby("user_id"):
        g = g.sort_values("utc_time") if "utc_time" in g.columns else g
        seq  = g["poi_idx"].tolist()
        hrs  = g["hour"].tolist()
        dows = g["dow"].tolist()
        splt = g["split"].tolist()

        # running counters over the PREFIX seq[:i] -> the profile can only ever see the past
        poi_c, cat_c, hr_c = Counter(), Counter(), Counter()
        poi_c[seq[0]] += 1; cat_c[poi_cat.get(seq[0], "Venue")] += 1; hr_c[hrs[0]] += 1

        for i in range(1, len(seq)):
            if splt[i] == target_split:                 # target must be in this split ...
                h = seq[max(0, i - hist_len):i]         # ... history spans the full trajectory
                ex.append(dict(
                    user       = uid,
                    hist       = h,
                    hist_hours = hrs[max(0, i - hist_len):i],
                    profile    = _profile_from_prefix(poi_c, cat_c, hr_c, i),   # <-- causal: from seq[:i]
                    target     = seq[i],
                    t_hour     = hrs[i],
                    t_dow      = dows[i],
                ))
            # only NOW does check-in i enter the counters, so it can never appear in its own profile
            poi_c[seq[i]] += 1
            cat_c[poi_cat.get(seq[i], "Venue")] += 1
            hr_c[hrs[i]] += 1
    return ex

val_ex  = build_split_examples(full_df, "val")
test_ex = build_split_examples(full_df, "test")

# --- leakage assertion: profile counts must sum to <= n_seen (the prefix length)
for _ex in (val_ex[:1000] + test_ex[:1000]):
    assert sum(c for _, c in _ex["profile"]["top_pois"]) <= _ex["profile"]["n_seen"], "profile is not causal!"
print("causality check passed (profile counts never exceed the prefix length)")

print(f"examples  val={len(val_ex)}  test={len(test_ex)}   (test is evaluated in FULL)")


## 4 · Hyperbolic → Euclidean projection utilities *(verbatim)*

In [ ]:
# ── logmap0 + projection utilities ─────────────────────────────────────────
import torch

def logmap0(x: torch.Tensor, c: float = 1.0, eps: float = 1e-9) -> torch.Tensor:
    """Poincare-ball log map at the origin. x: (N, d) with ||x|| < 1/sqrt(c)."""
    sqrt_c = c ** 0.5
    norm = x.norm(dim=-1, keepdim=True).clamp_min(eps)
    max_norm = (1.0 - 1e-5) / sqrt_c            # clamp inside the ball for numerical safety
    x = torch.where(norm > max_norm, x / norm * max_norm, x)
    norm = x.norm(dim=-1, keepdim=True).clamp_min(eps)
    return torch.atanh((sqrt_c * norm).clamp(max=1 - 1e-7)) * x / (sqrt_c * norm)

def build_injection(embs_np, d_model, condition, target_row_norm, seed=SEED):
    """Return (N_POI, d_model) fp32 rows to load into the frozen W_POI table."""
    g = torch.Generator().manual_seed(seed)
    if condition == "random":
        inj = torch.randn(embs_np.shape[0], d_model, generator=g)
    else:
        e = torch.tensor(embs_np, dtype=torch.float32)
        v = logmap0(e, c=CURVATURE_C) if condition == "hyperbolic" else e
        W = torch.randn(v.shape[1], d_model, generator=g) / (v.shape[1] ** 0.5)  # fixed, shared
        inj = v @ W
    inj = inj / inj.norm(dim=-1, keepdim=True).clamp_min(1e-9) * target_row_norm
    return inj  # (N_POI, d_model), float32

## 5 · transformers compatibility shim *(verbatim)*

In [ ]:
# Compat shim — direct attribute patch, no __init__ monkey-patching needed.
# LLaDA-MoE's cached modeling code already handles __init__ correctly on this
# transformers version; we just ensure the attribute exists post-construction
# by patching it onto the class as a default, not via __init__ wrapping.
import transformers.modeling_utils as _mu
if not hasattr(_mu.PreTrainedModel, "all_tied_weights_keys"):
    _mu.PreTrainedModel.all_tied_weights_keys = {}
print("compat shim installed")

## 6 · Load model (4-bit) + extend vocabulary *(verbatim)*

In [ ]:
# ── Model + tokenizer + vocab extension ────────────────────────────────────
import os, gc, time
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
import torch

# Load from the LOCAL snapshot fetched in §0a (pre-pin, over the fast hub). With the weights
# already on disk, from_pretrained does ZERO network I/O — so if this cell is still slow, the
# cost is compute (quantization / resize), not download. The per-phase timers below say which.
MODEL_SRC = globals().get("MODEL_PATH", MODEL_NAME)
if "MODEL_PATH" not in globals():
    print("⚠ MODEL_PATH not set — §0a did not run. Falling back to the hub id; expect the slow "
          "download over the downgraded huggingface_hub.")
print(f"loading from: {MODEL_SRC}")

# --- Free any model/optimizer left on the GPU by a PREVIOUS run of this cell ---
# Two distinct leak sources, both handled here:
#  1. Named globals (model, opt, ...) from a normal re-run without a kernel restart.
#  2. IPython's exception history: if a PREVIOUS cell was stopped mid-execution
#     (e.g. the training loop, interrupted via the Stop button rather than a kernel
#     restart), the raised KeyboardInterrupt's traceback is cached in sys.last_traceback
#     / sys.last_value. That traceback holds every local variable of every frame on the
#     stack at interrupt time (loss, hidden states, gradients, ...) — `del`-ing named
#     globals does NOT reach these, because they were never global. This is a well-known
#     Jupyter+CUDA gotcha: "Stop" does not reliably free GPU memory; only clearing the
#     exception history (or a full kernel restart) does.
for _v in ["model", "poi_head", "opt", "W_POI", "base_emb", "trainer"]:
    if _v in globals():
        try: del globals()[_v]
        except Exception: pass
import sys
for _attr in ("last_traceback", "last_value", "last_type", "last_exc"):
    if hasattr(sys, _attr):
        try: setattr(sys, _attr, None)
        except Exception: pass
try:
    _ip = get_ipython()
    if _ip is not None:
        _ip.last_traceback = None
except Exception:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache(); torch.cuda.ipc_collect()
    free, total = torch.cuda.mem_get_info()
    print(f"GPU free before load: {free/1e9:.2f} / {total/1e9:.2f} GiB")
    # The 4-bit model + tokenizer + resize needs ~6-7 GiB headroom. If we're still
    # below that after the cleanup above, memory is NOT actually free — restart
    # the KAGGLE SESSION (not just this cell) rather than re-running this cell again;
    # `del` and cache-clearing cannot reclaim CUDA memory pinned by a genuinely alive
    # Python object (e.g. a still-running DataLoader worker, or a second kernel).
    assert free > 6e9, (
        f"Only {free/1e9:.2f} GiB free — likely still held by a previous run. "
        "Restart the Kaggle SESSION (Session menu -> Restart), not just this cell, then re-run top to bottom."
    )

# T4 (Turing, sm_75) has NO native bf16 tensor cores → bf16 compute runs slow/emulated.
# Pick bf16 only on Ampere+ (sm_80+); otherwise fp16. This is the single biggest
# wall-clock fix on Kaggle T4s.
_bf16_ok = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
COMPUTE_DTYPE = torch.bfloat16 if _bf16_ok else torch.float16
print(f"GPU: {torch.cuda.get_device_name(0)}  compute_dtype={COMPUTE_DTYPE}")

# ── PHASE TIMERS ────────────────────────────────────────────────────────────
# If run 2 is STILL slow after the §0a fix, whichever number below is large tells us where
# the time actually goes — no more guessing at a cell that "hangs".
_t = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_SRC, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"[phase] tokenizer load        : {time.time()-_t:6.1f}s")

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=COMPUTE_DTYPE,
                         bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)

# MoE 4-bit fits on one T4; pin to cuda:0 and keep it simple.
_t = time.time()
model = AutoModel.from_pretrained(MODEL_SRC, trust_remote_code=True,
                                  quantization_config=bnb,
                                  device_map={"": 0})
print(f"[phase] from_pretrained (4bit): {time.time()-_t:6.1f}s   <- download + quantize")
print("device map:", getattr(model, "hf_device_map", "single-device"))
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"GPU free after model load: {free/1e9:.2f} / {total/1e9:.2f} GiB")

# --- register POI tokens & resize (reserves the id range; base rows unused at runtime) ---
_t = time.time()
poi_tokens = [f"<poi_{i}>" for i in range(N_POI)]
n_added = tokenizer.add_tokens(poi_tokens, special_tokens=True)
print(f"[phase] add_tokens ({n_added:5d})    : {time.time()-_t:6.1f}s")

_t = time.time()
model.resize_token_embeddings(len(tokenizer), mean_resizing=False)
print(f"[phase] resize_token_embeddings:{time.time()-_t:6.1f}s")

POI_TOKEN_IDS = torch.tensor(tokenizer.convert_tokens_to_ids(poi_tokens))
POI_ID_START  = int(POI_TOKEN_IDS.min())
POI_ID_END    = int(POI_TOKEN_IDS.max())
# sanity: the POI ids must be a contiguous block (the paper's [|V|, |V|+|P|-1] assumption)
assert POI_ID_END - POI_ID_START + 1 == N_POI, "POI token ids are not contiguous!"
assert (POI_TOKEN_IDS == torch.arange(POI_ID_START, POI_ID_END + 1)).all()
print(f"POI id range: [{POI_ID_START}, {POI_ID_END}]  (contiguous, N={N_POI})")

## 7 · Frozen `W_POI` table + lookup-time substitution *(verbatim — must match training exactly, it is deterministic given EMB_FILE/EMB_CONDITION/SEED)*

In [ ]:
# ── Build frozen W_POI and install the lookup-time substitution wrapper ────
import torch.nn as nn

base_emb = model.get_input_embeddings()          # discovered, not assumed
d_model  = base_emb.weight.shape[1]
emb_device = base_emb.weight.device

with torch.no_grad():
    # mean row-norm over the ORIGINAL vocab (exclude the freshly-resized POI rows)
    orig_rows = base_emb.weight[:POI_ID_START].float()
    tgt_norm  = orig_rows.norm(dim=-1).mean().item()
    inj = build_injection(poi_embs, d_model, EMB_CONDITION, tgt_norm)   # (N_POI, d_model) fp32

# separate, full-precision, FROZEN POI table
W_POI = nn.Embedding(N_POI, d_model, dtype=torch.float16, device=emb_device)
with torch.no_grad():
    W_POI.weight.copy_(inj.to(torch.float16))
W_POI.weight.requires_grad_(False)
print(f"W_POI: {tuple(W_POI.weight.shape)}  frozen={not W_POI.weight.requires_grad}  "
      f"norm≈{tgt_norm:.3f}  device={emb_device}")

# --- wrap the base embedding forward: POI ids -> W_POI, others -> base table ---
_POI_START = POI_ID_START
_POI_END   = POI_ID_END
_N_POI     = N_POI
_orig_emb_forward = base_emb.forward

def _mixed_embedding_forward(input_ids):
    # Route ONLY the contiguous POI id block to W_POI. Bounding on BOTH sides is essential:
    # the LLaDA-MoE mask id (156895) and other specials can sit near the top of the base vocab,
    # so a one-sided `>= START` check could misroute them. `[_POI_START, _POI_END]` is exact.
    is_poi   = (input_ids >= _POI_START) & (input_ids <= _POI_END)
    base_ids = torch.where(is_poi, torch.zeros_like(input_ids), input_ids)
    out = _orig_emb_forward(base_ids)                       # (B, L, d) from 4-bit base table
    if is_poi.any():
        poi_local = (input_ids[is_poi] - _POI_START).clamp_(0, _N_POI - 1)
        poi_vecs  = W_POI(poi_local).to(out.dtype)          # fp16 -> compute dtype
        out = out.clone()
        out[is_poi] = poi_vecs
    return out

base_emb.forward = _mixed_embedding_forward
print("mixed-embedding wrapper installed (POI ids served from frozen W_POI)")

## 8 · Load the **trained** LoRA adapter (inference only)

Training used `get_peft_model` on a fresh config; here we load the saved adapter with
`PeftModel.from_pretrained` instead — same wrapping, trained weights, nothing trainable.
The frozen-`W_POI` embedding wrapper installed in §7 must survive the PEFT wrap.


In [ ]:
# ── Load the TRAINED LoRA adapter from the checkpoint ───────────────────────
from peft import PeftModel

model = PeftModel.from_pretrained(model, CKPT_DIR, is_trainable=False)
model.eval()
if hasattr(model.config, "use_cache"):
    model.config.use_cache = False

# re-acquire the (now PEFT-wrapped) embedding module and re-assert the wrapper is intact
_emb_after = model.get_input_embeddings()
assert _emb_after.forward is _mixed_embedding_forward or getattr(_emb_after, "forward", None) is _mixed_embedding_forward \
       or _emb_after is base_emb, "embedding wrapper lost after PEFT wrap"

_n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"adapter loaded from: {CKPT_DIR}")
print(f"trainable params after load: {_n_trainable} (expected 0)")
print("W_POI still frozen:", not W_POI.weight.requires_grad)


## 9 · Rebuild the output scorer and load the **trained** head weights

`poi_head.pt` stores `scoring_mode` + the state dicts of `poi_norm` / `poi_head` / `poi_proj`.
`SCORING_MODE` is read **from the checkpoint** (not the config) so eval always ranks with the
exact scorer that was trained — the training notebook's own warning: loading only the adapter
(or ranking on `lm_head`'s untrained POI rows) gives ~random metrics.


In [ ]:
# ── Output scorer: rebuild modules, load trained weights, freeze ────────────
import torch.nn as nn

head_ckpt = torch.load(os.path.join(CKPT_DIR, "poi_head.pt"), map_location="cpu")
SCORING_MODE = head_ckpt.get("scoring_mode", "both")
print(f"SCORING_MODE (from checkpoint): {SCORING_MODE!r}")

H = model.config.hidden_size
PROJ_H = PROJ_HIDDEN or (H // 2)

poi_norm = nn.LayerNorm(H)
poi_head = nn.Linear(H, N_POI, bias=True)
poi_proj = nn.Sequential(
    nn.Linear(H, PROJ_H),
    nn.GELU(),
    nn.LayerNorm(PROJ_H),
    nn.Linear(PROJ_H, d_model, bias=False),
)

poi_norm.load_state_dict(head_ckpt["poi_norm"])
poi_head.load_state_dict(head_ckpt["poi_head"])
poi_proj.load_state_dict(head_ckpt["poi_proj"])
for _m in (poi_norm, poi_head, poi_proj):
    _m.to(emb_device, dtype=torch.float32).eval()
    for _p in _m.parameters():
        _p.requires_grad_(False)

# loaded-not-fresh sanity: the trained head cannot be all-zeros (zero-init was train-time only)
assert poi_head.weight.abs().sum().item() > 0, "poi_head is all zeros — wrong/empty checkpoint?"
if SCORING_MODE in ("tied", "both"):
    assert poi_proj[3].weight.abs().sum().item() > 0, \
        "tied-MLP last layer is still zero — this checkpoint never trained the tied path?"

_W_POI_f = W_POI.weight.detach().float()           # (N_POI, d_model), FROZEN

def poi_scores(h):
    """h: (num_targets, H) float -> (num_targets, N_POI) logits, per SCORING_MODE."""
    h = poi_norm(h)                                # scale control; identical to training
    if SCORING_MODE == "head":
        return poi_head(h)
    tied = poi_proj(h) @ _W_POI_f.t()              # depends on the frozen injected embeddings
    if SCORING_MODE == "tied":
        return tied
    return poi_head(h) + tied                      # "both" (run-2 default)

print(f"scorer ready: tied MLP {H} -> {PROJ_H} -> {d_model}  |  head Linear({H}, {N_POI})")


## 10 · Prompt + collator — *identical to training (left-pad with EOS, single masked target)*

In [ ]:
# ── prompt building (WITH user profile) + collator (LEFT-PAD with EOS) ─────
EOS_ID = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else (tokenizer.pad_token_id or 0)

def profile_text(pr):
    """Render the CAUSAL profile snapshot built in §3. Reads pr only — never the target."""
    if not pr["top_pois"]:
        return "[user profile]\nnew user, no prior check-ins\n"
    pois = ", ".join(f"<poi_{p}> ({poi_cat.get(p,'Venue')}, x{c})" for p, c in pr["top_pois"])
    lines = [f"[user profile]",
             f"check-ins so far: {pr['n_seen']}",
             f"most visited: {pois}"]
    if pr["top_cats"]:
        lines.append("favourite categories: " + ", ".join(map(str, pr["top_cats"])))
    if pr["top_hrs"]:
        lines.append("usual hours: " + ", ".join(f"{h}:00" for h in sorted(pr["top_hrs"])))
    return "\n".join(lines) + "\n"

def prompt_text(ex):
    hist_lines = "\n".join(f"<poi_{p}> ({poi_cat.get(p,'Venue')}, {h}:00)"
                           for p, h in zip(ex["hist"], ex["hist_hours"]))
    head = ("You are a POI recommendation expert. Using the user's long-term profile and their "
            "recent check-ins, predict the next POI token.\n")
    prof = profile_text(ex["profile"]) if USE_PROFILE else ""
    return (head + prof +
            "[recent check-ins]\n" + hist_lines +
            f"\n[current time] {ex['t_dow']} {ex['t_hour']}:00\n[next POI] ")

def encode_example(ex):
    p_ids = tokenizer(prompt_text(ex), add_special_tokens=False,
                      truncation=True, max_length=MAX_LEN - 2)["input_ids"]
    tgt_id = POI_ID_START + ex["target"]
    return p_ids, tgt_id

def collate(batch):
    enc = [encode_example(ex) for ex in batch]
    # row layout: [EOS pad ...][prompt tokens][MASK]   (target flush at the far right)
    L = max(len(p) + 1 for p, _ in enc)
    input_ids = torch.full((len(batch), L), EOS_ID, dtype=torch.long)   # EOS filler, not id-0
    labels    = torch.full((len(batch), L), -100,   dtype=torch.long)
    attn      = torch.ones((len(batch), L),          dtype=torch.long)  # passed but ignored by LLaDA
    for i, (p_ids, tgt) in enumerate(enc):
        n = len(p_ids)
        start = L - (n + 1)                              # LEFT-pad: real content flush-right
        input_ids[i, start:start + n] = torch.tensor(p_ids)
        input_ids[i, start + n]       = MASK_TOKEN_ID    # target position: always masked (N'=1)
        labels[i, start + n]          = tgt
    return dict(input_ids=input_ids, labels=labels, attention_mask=attn)

# --- eyeball one prompt + check the profile did not blow the token budget ---
_probe = test_ex[0]
print(prompt_text(_probe))
_lens = [len(encode_example(e)[0]) for e in test_ex[:200]]
print(f"prompt tokens over 200 examples: mean={np.mean(_lens):.0f}  max={max(_lens)}  (MAX_LEN={MAX_LEN})")
assert max(_lens) < MAX_LEN - 2, "prompts are hitting the truncation cap — lower HIST_LEN or PROFILE_TOP_K"

## 11 · Evaluation — restricted-logit ranking → **Acc@1 / Acc@5 / Acc@10 / MRR**

One forward pass per example; the trained scorer ranks the hidden state at the masked target
position over all POIs. Results are printed and written to a JSON in `/kaggle/working`.

Note: `val` here is the **full** re-split val set — during training the in-loop val may have been
a `VAL_MAX` subsample, so small differences vs the training log are expected. `test` is exact.


In [ ]:
# ── evaluation ──────────────────────────────────────────────────────────────
from torch.utils.data import DataLoader
import time

def _dev():
    return next(iter(model.parameters())).device   # (no requires_grad filter — everything is frozen here)

def _backbone():
    # The transformer stack WITHOUT the lm_head — skips the ~163k-vocab output projection.
    # LoRA adapters are module-level so they still apply; the W_POI wrapper also stays live.
    base = model.get_base_model() if hasattr(model, "get_base_model") else model
    return base.model

def _hidden_states(input_ids):
    out = _backbone()(input_ids=input_ids)
    return out.last_hidden_state if hasattr(out, "last_hidden_state") else out[0]   # (B, L, H)

@torch.no_grad()
def evaluate(examples, name="test"):
    model.eval()
    dev = _dev()
    loader = DataLoader(examples, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)
    ranks, t0 = [], time.time()
    for bi, batch in enumerate(loader):
        ids    = batch["input_ids"].to(dev)
        labels = batch["labels"].to(dev)
        h      = _hidden_states(ids)                       # (B, L, H); attn mask ignored by LLaDA-MoE
        pos    = (labels != -100)
        poi_logits = poi_scores(h[pos].float())            # (num, N_POI) via the TRAINED scorer
        tgt    = labels[pos] - POI_ID_START
        r = (poi_logits > poi_logits.gather(1, tgt[:, None])).sum(1) + 1
        ranks.extend(r.tolist())
        if bi % 50 == 0:
            print(f"  [{name}] batch {bi}/{len(loader)}  ({time.time()-t0:.0f}s)")
    ranks = torch.tensor(ranks, dtype=torch.float)
    res = dict(n=len(ranks),
               acc1=(ranks <= 1).float().mean().item(),
               acc5=(ranks <= 5).float().mean().item(),
               acc10=(ranks <= 10).float().mean().item(),
               mrr=(1.0 / ranks).mean().item())
    print(f"[{name}] n={res['n']}  Acc@1={res['acc1']:.4f}  Acc@5={res['acc5']:.4f}  "
          f"Acc@10={res['acc10']:.4f}  MRR={res['mrr']:.4f}")
    return res

results = dict(model=MODEL_NAME, ckpt_repo=CKPT_REPO, ckpt_subfolder=CKPT_SUBFOLDER,
               condition=EMB_CONDITION, scoring_mode=SCORING_MODE)
if EVAL_VAL:
    results["val"] = evaluate(val_ex, "val")
results["test"] = evaluate(test_ex, "test")

import json
_tag = CKPT_REPO.split("/")[-1]
out_json = f"{OUT_DIR}/test_results_{DATASET}_{EMB_CONDITION}_{_tag}_{CKPT_SUBFOLDER}.json"
with open(out_json, "w") as f:
    json.dump(results, f, indent=2)
print("saved:", out_json)


## Checklist if the numbers look wrong

- **~Random metrics (Acc@1 ≈ 1/5120)** → the head or adapter didn't load: §8 must print
  `trainable params after load: 0`, §9 must print the checkpoint's `scoring_mode` and pass both
  all-zero asserts.
- **Plausible-but-low metrics** → `EMB_CONDITION` / `EMB_FILE` don't match what this checkpoint was
  trained with (the tied path scores against the wrong `W_POI`), or the split fractions / `SEED`
  / `HIST_LEN` / profile knobs differ from the training run.
- The collator cell prints one full test prompt — **read it**: the `[user profile]` block must
  only contain POIs the user visited before the target.
- `MASK_TOKEN_ID` must be **156895** (LLaDA-MoE), and §7 must print `W_POI still frozen: True`.
- `val` may differ slightly from the training log (see §11 note); `test` is the exact test set.
